# SentinelGrafo — SOFTWARE DOCUMENTADO

**Clasificación de mensajes críticos en plataformas digitales mediante Redes Neuronales Gráficas**

---

## Índice del Software Documentado

1. Instalación de dependencias
2. Definición de rutas (auto-detección)
3. Pipeline común: limpieza de texto y construcción de grafos
4. Carga y preprocesamiento: Caso 1 — Disaster Tweets
5. Carga y preprocesamiento: Caso 2 — AG News (4 categorías)
6. Visualización de la topología de los grafos
7. Baseline: Logistic Regression sobre vectores TF-IDF
8. Definición de modelos GNN: GraphSAGE y GCN
9. Función unificada de entrenamiento y evaluación
10. Entrenamiento de modelos para ambos casos
11. Visualización 1: Curvas de entrenamiento (pérdida y accuracy)
12. Visualización 2: Matrices de confusión
13. Visualización 3: Proyección t-SNE de embeddings
14. Visualización 4: Diagrama esquemático de capas GNN (Message Passing)
15. Visualización 5: Gráfico de barras comparativo (Baseline vs GNN)
16. Dashboard interactivo — Demo de predicción en vivo
17. Tabla resumen final de resultados

---


## 1. Instalación de dependencias

Se instalan las bibliotecas necesarias:
- `torch_geometric` para GNN sobre grafos
- `ipywidgets` para el dashboard interactivo
- `seaborn` para visualizaciones estadísticas
- `scikit-learn` para TF-IDF, baseline y métricas
- `networkx` y `matplotlib` para visualización de grafos

In [ ]:
import sys
import subprocess

def install_package(package_name, alias=None):
    display_name = alias if alias else package_name
    print(f'Instalando {display_name}...')
    try:
        __import__(package_name.split('==')[0].split('>')[0].replace('-','_'))
        print(f'{display_name} ya esta instalado.')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name, '-q'])
        print(f'{display_name} instalado correctamente.')

install_package('torch_geometric', 'PyTorch Geometric')
install_package('ipywidgets', 'ipywidgets')
install_package('seaborn', 'Seaborn')
install_package('networkx', 'NetworkX')
install_package('plotly', 'Plotly')

print('\nTodas las dependencias listas.')

## 1b. Importación de todas las bibliotecas

Se centralizan todos los imports para mantener el notebook ordenado.

In [ ]:
import pandas as pd
import numpy as np
import re
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GCNConv
from torch_geometric.utils import to_networkx, k_hop_subgraph

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
sns.set_style('whitegrid')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
print(f'Dispositivo: {DEVICE}')

## 2. Definición de rutas (auto-detección)

El notebook detecta automáticamente la raíz del proyecto buscando la carpeta `03_Datasets`.
No es necesario ajustar rutas manualmente: funciona en la máquina de cualquier integrante.

In [ ]:
import os
from pathlib import Path

# Auto-deteccion de la raiz del proyecto
_current = Path.cwd().resolve()
BASE_PATH = None
for _ in range(5):
    if (_current / '03_Datasets').exists():
        BASE_PATH = str(_current)
        print(f'Raiz del proyecto detectada: {BASE_PATH}')
        break
    _current = _current.parent

if BASE_PATH is None:
    raise FileNotFoundError(
        'No se encontro la carpeta 03_Datasets. '
        'Ejecuta el notebook desde dentro de la carpeta del proyecto.'
    )

RUTA_DISASTER_TRAIN = os.path.join(BASE_PATH, '03_Datasets', 'disaster_tweets', 'train.csv')
RUTA_AG_NEWS      = os.path.join(BASE_PATH, '03_Datasets', 'ag_news', 'ag_news_train.csv')

for name, path in [('Disaster Train', RUTA_DISASTER_TRAIN),
                    ('AG News', RUTA_AG_NEWS)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f'[{name}] Encontrado ({size_mb:.1f} MB): {path}')
    else:
        print(f'[{name}] NO ENCONTRADO: {path}')


## 3. Pipeline común — Limpieza de texto y construcción de grafos

Ambos casos de estudio comparten el mismo pipeline de preprocesamiento.
Se definen dos funciones reutilizables:

- **`clean_text(text)`** — Normaliza el texto: minúsculas, elimina URLs,
  HTML, puntuación y espacios sobrantes.
- **`build_graph(texts, labels, max_features, k)`** — Vectoriza con TF-IDF,
  construye el grafo con similitud coseno y k vecinos más cercanos,
  y empaqueta todo en un objeto `torch_geometric.data.Data` listo para la GNN.

In [ ]:
def clean_text(text):
    '''Limpia y normaliza un texto para el pipeline de NLP.'''
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zñáéíóúü\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_graph(texts, labels, max_features=3000, k=5):
    '''
    Construye un grafo textual a partir de una lista de textos.
    Retorna: (torch_geometric.data.Data, TfidfVectorizer, np.array TF-IDF)
    '''
    print(f'Vectorizando {len(texts)} textos con TF-IDF (max_features={max_features})...')
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X_tfidf = vectorizer.fit_transform(texts).toarray()
    print(f'Matriz TF-IDF: {X_tfidf.shape}')

    from sklearn.neighbors import NearestNeighbors
    print(f'Buscando {k} vecinos mas cercanos con NearestNeighbors (coseno)...')
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', n_jobs=-1)
    knn.fit(X_tfidf)
    _, indices = knn.kneighbors(X_tfidf)
    neighbor_indices = indices[:, 1:]  # excluye self (posicion 0)

    n = len(texts)
    edge_src = np.repeat(np.arange(n), k)
    edge_dst = neighbor_indices.flatten()

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    x = torch.tensor(X_tfidf, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    print(f'Grafo construido: {data.num_nodes} nodos, {data.num_edges} aristas, '
          f'{data.num_features} features/nodo, {len(torch.unique(y))} clases')

    return data, vectorizer, X_tfidf


## 4. Caso 1 — Disaster Tweets: carga, balanceo y preprocesamiento

Se carga el dataset `train.csv` del desafío *Natural Language Processing with Disaster Tweets*.
Cada tweet tiene una etiqueta binaria:
- `target = 1` → el tweet describe un desastre real
- `target = 0` → el tweet NO describe un desastre real (uso figurado o irrelevante)

Se realiza balanceo de clases, limpieza de texto y split 80/20 para train/test.

In [ ]:
print('=' * 60)
print('CASO 1: DETECCION DE TWEETS SOBRE DESASTRES REALES')
print('=' * 60)

df_tweets = pd.read_csv(RUTA_DISASTER_TRAIN)
print(f'Tweets cargados: {len(df_tweets)}')
print(f'Columnas: {list(df_tweets.columns)}')
print(f'Distribucion de clases:\n{df_tweets["target"].value_counts()}')

df_tweets['clean_text'] = df_tweets['text'].apply(clean_text)
df_tweets = df_tweets[df_tweets['clean_text'].str.len() > 0].reset_index(drop=True)

min_class = df_tweets['target'].value_counts().min()
df_c1 = pd.concat([
    df_tweets[df_tweets['target'] == 0].sample(n=min_class, random_state=42),
    df_tweets[df_tweets['target'] == 1].sample(n=min_class, random_state=42)
], ignore_index=True)
df_c1 = df_c1.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nDataset balanceado Caso 1: {len(df_c1)} muestras')
print(f'Distribucion: {dict(df_c1["target"].value_counts())}')

texts_c1  = df_c1['clean_text'].tolist()
labels_c1 = df_c1['target'].tolist()

data_c1, vec_c1, tfidf_c1 = build_graph(texts_c1, labels_c1, max_features=3000, k=5)

idx_c1 = np.arange(len(df_c1))
train_idx_c1, test_idx_c1 = train_test_split(idx_c1, test_size=0.2,
                                            random_state=42, stratify=labels_c1)
train_mask_c1 = torch.zeros(data_c1.num_nodes, dtype=torch.bool)
test_mask_c1  = torch.zeros(data_c1.num_nodes, dtype=torch.bool)
train_mask_c1[train_idx_c1] = True
test_mask_c1[test_idx_c1]   = True
print(f'\nSplit Caso 1 — Train: {train_mask_c1.sum().item()}, Test: {test_mask_c1.sum().item()}')

## 5. Caso 2 — AG News: carga, balanceo y preprocesamiento

Se carga el dataset **AG News**, un conjunto de noticias clasificadas en 4 categorías:
- **World** (Mundo): noticias internacionales y política
- **Sports** (Deportes): noticias deportivas
- **Business** (Negocios): noticias económicas y financieras
- **Sci/Tech** (Ciencia/Tecnología): noticias de ciencia y tecnología

Cada noticia tiene un título (`title`) y una descripción (`description`).
Se concatenan en un solo campo `content`.
Se muestrean 6,000 noticias por clase (24,000 totales) para mantener baja complejidad.
Luego se limpia, vectoriza con TF-IDF, construye el grafo k-NN y se hace split 80/20.


In [ ]:
print('=' * 60)
print('CASO 2: CLASIFICACION DE NOTICIAS — AG NEWS')
print('=' * 60)

MUESTRAS_POR_CLASE = 6000

print('Cargando archivo AG News...')
df_ag = pd.read_csv(RUTA_AG_NEWS, header=None,
                    names=['label', 'title', 'description'])
df_ag['label'] = pd.to_numeric(df_ag['label']) - 1  # '1'..'4' -> 0..3
print(f'Noticias cargadas: {len(df_ag)}')
print(f'Distribucion de clases:\n{df_ag["label"].value_counts().sort_index()}')

# Muestreo balanceado
dfs_c2 = []
for cls in range(4):
    df_cls = df_ag[df_ag['label'] == cls]
    dfs_c2.append(df_cls.sample(n=min(MUESTRAS_POR_CLASE, len(df_cls)), random_state=42))
df_c2 = pd.concat(dfs_c2, ignore_index=True)
df_c2 = df_c2.sample(frac=1, random_state=42).reset_index(drop=True)

df_c2['content'] = df_c2['title'].fillna('') + ' ' + df_c2['description'].fillna('')
df_c2['clean_content'] = df_c2['content'].apply(clean_text)
df_c2 = df_c2[df_c2['clean_content'].str.len() > 0].reset_index(drop=True)

print(f'\nDataset Caso 2: {len(df_c2)} noticias (6000 x 4 clases)')
print(f'Distribucion final: {dict(df_c2["label"].value_counts())}')

texts_c2  = df_c2['clean_content'].tolist()
labels_c2 = df_c2['label'].tolist()

data_c2, vec_c2, tfidf_c2 = build_graph(texts_c2, labels_c2, max_features=5000, k=5)

idx_c2 = np.arange(len(df_c2))
train_idx_c2, test_idx_c2 = train_test_split(idx_c2, test_size=0.2,
                                            random_state=42, stratify=labels_c2)
train_mask_c2 = torch.zeros(data_c2.num_nodes, dtype=torch.bool)
test_mask_c2  = torch.zeros(data_c2.num_nodes, dtype=torch.bool)
train_mask_c2[train_idx_c2] = True
test_mask_c2[test_idx_c2]   = True
print(f'\nSplit Caso 2 — Train: {train_mask_c2.sum().item()}, Test: {test_mask_c2.sum().item()}')


## 6. Visualización de la topología de los grafos

Se visualiza un subconjunto de 800 nodos de cada grafo para apreciar la estructura
de conexiones generada por la similitud textual.

**Caso 1 (Disaster Tweets):** Naranja = Desastre real, Azul = No desastre
**Caso 2 (AG News):** 4 colores para las 4 categorias (World, Sports, Business, Sci/Tech)


In [ ]:
def plot_graph_subset(data, labels_tensor, title, colors, labels,
                      max_nodes=800, seed=42):
    '''Visualiza un subgrafo cohesionado de max_nodes seleccionado por BFS para mostrar la estructura real.'''
    from collections import defaultdict

    # 1. Construir lista de adyacencia del grafo completo
    edge_index = data.edge_index
    edge_src = edge_index[0].cpu().numpy()
    edge_dst = edge_index[1].cpu().numpy()

    adj = defaultdict(list)
    for u, v in zip(edge_src, edge_dst):
        adj[u].append(v)
        adj[v].append(u)

    # 2. Seleccionar nodos usando BFS para garantizar que estén conectados
    start_node = max(adj.keys(), key=lambda x: len(adj[x]))

    selected_nodes = set()
    queue = [start_node]

    random.seed(seed)

    while queue and len(selected_nodes) < max_nodes:
        curr = queue.pop(0)
        if curr not in selected_nodes:
            selected_nodes.add(curr)
            for neighbor in adj[curr]:
                if neighbor not in selected_nodes and neighbor not in queue:
                    queue.append(neighbor)

        # Si la componente se agota, tomamos otra semilla del residuo
        if not queue and len(selected_nodes) < max_nodes:
            remaining = set(range(data.num_nodes)) - selected_nodes
            if remaining:
                remaining_adj = {n: adj[n] for n in remaining if n in adj}
                if remaining_adj:
                    next_seed = max(remaining_adj.keys(), key=lambda x: len(remaining_adj[x]))
                else:
                    next_seed = random.choice(list(remaining))
                queue.append(next_seed)
            else:
                break

    # 3. Crear el subgrafo inducido por los nodos seleccionados
    node_list = sorted(list(selected_nodes))
    node_to_idx = {old_id: new_id for new_id, old_id in enumerate(node_list)}

    sub_edges_src = []
    sub_edges_dst = []
    node_set = set(node_list)
    for u in node_list:
        for v in adj[u]:
            if v in node_set and u < v:
                sub_edges_src.append(node_to_idx[u])
                sub_edges_dst.append(node_to_idx[v])

    sub_ei = torch.tensor([sub_edges_src, sub_edges_dst], dtype=torch.long)
    sub_x = data.x[node_list]

    if torch.is_tensor(labels_tensor):
        y_np = labels_tensor.cpu().numpy()
    else:
        y_np = np.array(labels_tensor)

    sub_y = torch.tensor(y_np[node_list], dtype=torch.long)
    sub_data = Data(x=sub_x, edge_index=sub_ei, y=sub_y)
    G = to_networkx(sub_data, to_undirected=True)

    node_colors = [colors[lbl] for lbl in y_np[node_list]]

    # 4. Dibujar el grafo de forma atractiva
    fig, ax = plt.subplots(figsize=(12, 10))
    pos = nx.spring_layout(G, seed=seed, k=0.08, iterations=100)

    # Aristas con opacidad suave
    nx.draw_networkx_edges(G, pos, alpha=0.12, edge_color='gray', ax=ax)
    # Nodos
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=35,
                           alpha=0.9, ax=ax)

    legend_elements = [mpatches.Patch(color=colors[i], label=labels[i])
                       for i in range(len(colors))]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10, frameon=True)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


print('\n>>> VISUALIZACION 1: TOPOLOGIA DE LOS GRAFOS <<<\n')

plot_graph_subset(data_c1, labels_c1,
                  'Caso 1: Grafo de Disaster Tweets (muestra de 800 nodos)',
                  colors=['#FF8C00', '#1E90FF'],
                  labels=['Desastre real', 'No desastre'])

plot_graph_subset(data_c2, labels_c2,
                  'Caso 2: Grafo de AG News (muestra de 800 nodos)',
                  colors=['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'],
                  labels=['World', 'Sports', 'Business', 'Sci/Tech'])


## 7. Baseline: Logistic Regression sobre vectores TF-IDF

Antes de entrenar las GNN, se establece un baseline con Regresión Logística
entrenada directamente sobre los vectores TF-IDF (sin información del grafo).

Esto permite medir cuánto aporta la estructura de grafo y el message passing de la GNN.

In [ ]:
def run_baseline(tfidf_matrix, labels, train_idx, test_idx, case_name):
    '''Entrena Logistic Regression sobre TF-IDF y retorna metricas.'''
    X_train = tfidf_matrix[train_idx]
    X_test  = tfidf_matrix[test_idx]
    y_train = np.array(labels)[train_idx]
    y_test  = np.array(labels)[test_idx]

    model = LogisticRegression(max_iter=2000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

    print(f'--- Baseline Logistic Regression: {case_name} ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    target_names = ['Clase 0', 'Clase 1'] if len(set(labels)) == 2 else ['World', 'Sports', 'Business', 'Sci/Tech']
    print(classification_report(y_test, y_pred,
          target_names=target_names))

    return {'model': model, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'y_pred': y_pred, 'y_true': y_test}


print('\n>>> EVALUACION BASELINE <<<\n')

baseline_c1 = run_baseline(tfidf_c1, labels_c1, train_idx_c1, test_idx_c1,
                           'Caso 1: Disaster Tweets')
print()
baseline_c2 = run_baseline(tfidf_c2, labels_c2, train_idx_c2, test_idx_c2,
                           'Caso 2: AG News')


## 8. Definición de modelos GNN: GraphSAGE y GCN

Se implementa una clase `GNNClassifier` unificada que acepta un parámetro `model_type`
para seleccionar la arquitectura:

- **GraphSAGE (`sage`)**: Utiliza `SAGEConv` de PyG. Aprende una función de agregación
  sobre los vecinos muestreados. Es inductivo: puede generalizar a nodos no vistos.
- **GCN (`gcn`)**: Utiliza `GCNConv` de PyG. Aplica una convolución espectral
  de primer orden sobre todo el grafo.

Ambos modelos comparten la misma estructura: 2 capas de convolución de grafo con
activación ReLU, dropout para regularización, y una capa lineal final de clasificación.

El método `get_embeddings()` extrae las representaciones latentes de la penúltima capa,
útiles para la visualización t-SNE.

In [ ]:
class GNNClassifier(nn.Module):
    '''
    Clasificador basado en Graph Neural Networks.
    Soporta GraphSAGE y GCN como backbones.
    '''
    def __init__(self, in_channels, hidden_channels, out_channels,
                 model_type='sage', dropout=0.5):
        super().__init__()
        self.model_type = model_type
        self.dropout = dropout
        self.hidden_channels = hidden_channels

        if model_type == 'sage':
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        elif model_type == 'gcn':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels)
        else:
            raise ValueError(f"Modelo '{model_type}' no soportado. Use 'sage' o 'gcn'.")

        self.classifier = nn.Linear(hidden_channels, out_channels)
        self._embeddings = None

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        self._embeddings = x
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.classifier(x)
        return x

    def get_embeddings(self):
        '''Retorna las embeddings de la penultima capa.'''
        return self._embeddings.detach().cpu().numpy()


print('Clase GNNClassifier definida correctamente.')
print('Arquitecturas soportadas: GraphSAGE (sage), GCN (gcn)')

## 9. Función unificada de entrenamiento y evaluación

`train_and_evaluate` entrena un modelo GNN con:
- Optimizador Adam con weight decay para regularización L2
- Cross-entropy loss
- Early stopping con paciencia configurable (restaura los mejores pesos)
- Registro de pérdida y accuracy por época para curvas de aprendizaje
- Extracción de embeddings finales para visualización t-SNE

In [ ]:
def train_and_evaluate(data, train_mask, test_mask, model_type='sage',
                       hidden_dim=64, epochs=200, lr=0.01, patience=30):
    '''
    Entrena un GNNClassifier y retorna modelo, historial, predicciones,
    embeddings y mejor accuracy en test.
    '''
    model = GNNClassifier(
        in_channels=data.num_features,
        hidden_channels=hidden_dim,
        out_channels=len(torch.unique(data.y)),
        model_type=model_type,
        dropout=0.5
    ).to(DEVICE)

    data = data.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': [], 'test_loss': []}
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    pbar_prefix = f'{model_type.upper()} '
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            train_pred = out_eval[train_mask].argmax(dim=1)
            train_acc = (train_pred == data.y[train_mask]).float().mean().item()
            test_pred = out_eval[test_mask].argmax(dim=1)
            test_acc = (test_pred == data.y[test_mask]).float().mean().item()
            test_loss_val = criterion(out_eval[test_mask], data.y[test_mask]).item()

        history['train_loss'].append(loss.item())
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['test_loss'].append(test_loss_val)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f'{pbar_prefix}Early stopping en epoca {epoch+1}/{epochs}')
            break
    else:
        print(f'{pbar_prefix}Entrenamiento completo: {epochs} epocas')

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        test_pred = out[test_mask].argmax(dim=1).cpu().numpy()
        test_true = data.y[test_mask].cpu().numpy()
        embeddings = model.get_embeddings()

    results = {
        'model': model,
        'history': history,
        'y_pred': test_pred,
        'y_true': test_true,
        'embeddings': embeddings,
        'best_acc': best_test_acc,
    }
    return results


print('Funcion train_and_evaluate lista.')

## 10. Entrenamiento de modelos para ambos casos

Se entrena **GraphSAGE** y **GCN** para cada caso de estudio.
Todos los resultados se almacenan en diccionarios para su análisis posterior.

**Modelos a entrenar:**
- Caso 1 (Disaster Tweets): GraphSAGE + GCN
- Caso 2 (AG News): GraphSAGE + GCN

**Total: 4 modelos**


In [ ]:
print('=' * 60)
print('ENTRENAMIENTO DE MODELOS GNN')
print('=' * 60)

all_results = {}

# -----------------------------------------------------------
# Caso 1 — Disaster Tweets
# -----------------------------------------------------------
print('\n>>> CASO 1: Disaster Tweets — GraphSAGE <<<')
r_c1_sage = train_and_evaluate(data_c1, train_mask_c1, test_mask_c1,
                                model_type='sage', hidden_dim=64, epochs=200)
all_results['C1_GraphSAGE'] = r_c1_sage

print('\n>>> CASO 1: Disaster Tweets — GCN <<<')
r_c1_gcn = train_and_evaluate(data_c1, train_mask_c1, test_mask_c1,
                               model_type='gcn', hidden_dim=64, epochs=200)
all_results['C1_GCN'] = r_c1_gcn

# -----------------------------------------------------------
# Caso 2 — AG News
# -----------------------------------------------------------
print('\n>>> CASO 2: AG News — GraphSAGE <<<')
r_c2_sage = train_and_evaluate(data_c2, train_mask_c2, test_mask_c2,
                                model_type='sage', hidden_dim=64, epochs=200)
all_results['C2_GraphSAGE'] = r_c2_sage

print('\n>>> CASO 2: AG News — GCN <<<')
r_c2_gcn = train_and_evaluate(data_c2, train_mask_c2, test_mask_c2,
                               model_type='gcn', hidden_dim=64, epochs=200)
all_results['C2_GCN'] = r_c2_gcn

print('\n' + '=' * 60)
print('ENTRENAMIENTO COMPLETADO')
print('=' * 60)

for name, res in all_results.items():
    acc = res['best_acc']
    yp = res['y_pred']
    yt = res['y_true']
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, average='macro')
    print(f'{name:20s} — Accuracy: {acc:.4f}  |  F1-macro: {f1:.4f}')


## 11. Visualización 1 — Curvas de entrenamiento

Se muestran las curvas de pérdida (loss) y accuracy durante el entrenamiento
para cada modelo. Esto permite diagnosticar overfitting, velocidad de convergencia
y comparar el comportamiento de GraphSAGE vs GCN.

In [ ]:
print('\n>>> VISUALIZACION 2: CURVAS DE ENTRENAMIENTO <<<\n')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

plot_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1: Disaster Tweets — GraphSAGE', '#FF8C00'),
    (0, 1, 'C1_GCN',       'Caso 1: Disaster Tweets — GCN',       '#1E90FF'),
    (1, 0, 'C2_GraphSAGE', 'Caso 2: AG News — GraphSAGE',  '#2E8B57'),
    (1, 1, 'C2_GCN',       'Caso 2: AG News — GCN',        '#DC143C'),
]

for row, col, key, title, color in plot_configs:
    h = all_results[key]['history']
    epochs = range(1, len(h['train_loss']) + 1)
    ax1 = axes[row, col]
    ax2 = ax1.twinx()

    ax1.plot(epochs, h['train_loss'], color=color, linestyle='-', linewidth=2,
             label='Train Loss')
    ax1.plot(epochs, h['test_loss'], color=color, linestyle='--', linewidth=2,
             label='Test Loss')
    ax2.plot(epochs, h['test_acc'], color='gray', linestyle='-.', linewidth=2,
             label='Test Accuracy')

    ax1.set_xlabel('Epoca', fontsize=10)
    ax1.set_ylabel('Loss', fontsize=10, color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    ax2.set_ylabel('Accuracy', fontsize=10, color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.set_ylim(0, 1.05)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)

    ax1.set_title(title, fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)

plt.suptitle('Curvas de Entrenamiento — Todos los Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 12. Visualización 2 — Matrices de confusión

Las matrices de confusión muestran cuántas predicciones acertó y falló cada modelo.
Se presentan en un grid 2x2 para comparación directa.

Para cada celda se incluye además el reporte de clasificación completo (precision, recall, F1).

In [ ]:
print('\n>>> VISUALIZACION 3: MATRICES DE CONFUSION <<<\n')

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

conf_labels_c1 = ['No desastre', 'Desastre real']
conf_labels_c2 = ['World', 'Sports', 'Business', 'Sci/Tech']

conf_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1 — GraphSAGE', conf_labels_c1, 'Oranges'),
    (0, 1, 'C1_GCN',       'Caso 1 — GCN',       conf_labels_c1, 'Blues'),
    (1, 0, 'C2_GraphSAGE', 'Caso 2 — GraphSAGE', conf_labels_c2, 'viridis'),
    (1, 1, 'C2_GCN',       'Caso 2 — GCN',       conf_labels_c2, 'plasma'),
]

for row, col, key, title, labels, cmap in conf_configs:
    r = all_results[key]
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[row, col],
                xticklabels=labels, yticklabels=labels, cbar=True,
                linewidths=0.5, linecolor='white')
    axes[row, col].set_title(title, fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel('Prediccion', fontsize=10)
    axes[row, col].set_ylabel('Real', fontsize=10)
    if len(labels) > 2:
        axes[row, col].tick_params(axis='x', rotation=45)

plt.suptitle('Matrices de Confusion — Todos los Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\nReportes de clasificacion detallados:\n')
for key, label_names in [('C1_GraphSAGE', conf_labels_c1),
                           ('C1_GCN', conf_labels_c1),
                           ('C2_GraphSAGE', conf_labels_c2),
                           ('C2_GCN', conf_labels_c2)]:
    r = all_results[key]
    print(f'--- {key} ---')
    print(classification_report(r['y_true'], r['y_pred'], target_names=label_names))
    print()


## 13. Visualización 3 — Proyección t-SNE de embeddings

Se extraen los embeddings de la penúltima capa de cada GNN (64 dimensiones)
y se proyectan a 2D usando t-SNE.

Esto permite observar si el modelo logra separar las clases en el espacio latente.
Los colores indican la clase real; los marcadores, la predicción del modelo.
Un círculo alrededor de un punto indica que el modelo acertó; una X indica error.

In [ ]:
print('\n>>> VISUALIZACION 4: PROYECCION t-SNE DE EMBEDDINGS <<<\n')

def plot_tsne(embeddings, y_true, y_pred, title, ax, colors, class_names):
    '''Proyecta embeddings con t-SNE y colorea por clase real + acierto/error.'''
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)//3))
    emb_2d = tsne.fit_transform(embeddings)

    correct = (y_true == y_pred)
    for cls_val in range(len(colors)):
        mask_correct = (y_true == cls_val) & correct
        mask_error = (y_true == cls_val) & ~correct
        if mask_correct.any():
            ax.scatter(emb_2d[mask_correct, 0], emb_2d[mask_correct, 1],
                       c=colors[cls_val], marker='o', s=30, alpha=0.7,
                       edgecolors='white', linewidth=0.3,
                       label=f'{class_names[cls_val]} (OK)')
        if mask_error.any():
            ax.scatter(emb_2d[mask_error, 0], emb_2d[mask_error, 1],
                       c=colors[cls_val], marker='X', s=60, alpha=0.9,
                       edgecolors='black', linewidth=0.5,
                       label=f'{class_names[cls_val]} (Error)')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=6, loc='best')


fig, axes = plt.subplots(2, 2, figsize=(18, 16))

tsne_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1 — GraphSAGE Embeddings',
     ['#FF8C00', '#1E90FF'], ['Desastre', 'No Desastre']),
    (0, 1, 'C1_GCN',       'Caso 1 — GCN Embeddings',
     ['#FF8C00', '#1E90FF'], ['Desastre', 'No Desastre']),
    (1, 0, 'C2_GraphSAGE', 'Caso 2 — GraphSAGE Embeddings',
     ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'],
     ['World', 'Sports', 'Business', 'Sci/Tech']),
    (1, 1, 'C2_GCN',       'Caso 2 — GCN Embeddings',
     ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'],
     ['World', 'Sports', 'Business', 'Sci/Tech']),
]

for row, col, key, title, colors, class_names in tsne_configs:
    r = all_results[key]
    mask_test = test_mask_c1.numpy() if 'C1' in key else test_mask_c2.numpy()
    plot_tsne(r['embeddings'][mask_test], r['y_true'], r['y_pred'],
              title, axes[row, col], colors, class_names)

plt.suptitle('Visualizacion t-SNE de Embeddings de las GNN',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 14. Visualización 4 — Diagrama esquemático de capas GNN (Message Passing)

Se dibuja un diagrama conceptual del mecanismo de **Message Passing** que usan
GraphSAGE y GCN. El flujo es:

1. **Nodos de entrada** con features (TF-IDF, 3000 dims)
2. **AGGREGATE**: cada nodo recibe mensajes de sus vecinos
3. **UPDATE**: se combina la información propia con la agregada (MLP)
4. **Nueva representación** (embedding, 64 dims)
5. **Clasificador**: capa lineal que produce logits para cada clase

Esto se repite en 2 capas de convolución para propagar información a 2 saltos de distancia.

In [ ]:
print('\n>>> VISUALIZACION 5: DIAGRAMA DE CAPAS GNN (MESSAGE PASSING) <<<\n')

def draw_gnn_diagram():
    fig, ax = plt.subplots(1, 1, figsize=(16, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 7)
    ax.axis('off')

    def draw_box(x, y, w, h, text, color, fontsize=10, text_color='white'):
        rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                       facecolor=color, edgecolor='black',
                                       linewidth=1.5, alpha=0.9)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color)

    def draw_arrow(x1, y1, x2, y2, label='', color='gray'):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2,
                                    connectionstyle='arc3,rad=0'))
        if label:
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.25
            ax.text(mx, my, label, ha='center', va='bottom',
                    fontsize=9, fontstyle='italic', color=color)

    y_center = 3.5

    draw_box(0.5, y_center-0.8, 2.8, 1.6,
             'NODOS DE\nENTRADA\n(features TF-IDF)', '#2C3E50', 10)

    draw_box(5.0, y_center-0.8, 2.8, 1.6,
             'AGGREGATE\n(Mensajes de\nvecinos k-NN)', '#2980B9', 10)

    draw_box(9.0, y_center-0.8, 2.8, 1.6,
             'UPDATE\n(MLP + ReLU\n+ Dropout)', '#27AE60', 10)

    draw_box(13.0, y_center-0.8, 2.8, 1.6,
             'EMBEDDINGS\n(64 dimensiones\nespacio latente)', '#8E44AD', 10)

    draw_box(6.5, 0.8, 4.5, 1.2,
             'CLASIFICADOR\nCapa Lineal -> Logits (2 clases)', '#E74C3C', 10)

    draw_arrow(3.3, y_center, 5.0, y_center, 'message', '#3498DB')
    draw_arrow(7.8, y_center, 9.0, y_center, 'passing', '#2ECC71')
    draw_arrow(11.8, y_center, 13.0, y_center, '', '#9B59B6')

    ax.annotate('', xy=(8.75, 1.8), xytext=(8.75, y_center-0.8),
                arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2,
                                connectionstyle='arc3,rad=0'))

    ax.annotate('', xy=(8.75, 0.6), xytext=(8.75, 2.0),
                arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2,
                                connectionstyle='arc3,rad=0.3'))
    ax.text(9.3, 1.3, 'x2 capas\n(2-hop\nmessage\npassing)',
            ha='left', va='center', fontsize=8, fontstyle='italic', color='#7F8C8D')

    ax.text(8, 6.3, 'DIAGRAMA DE MESSAGE PASSING EN GNN (GraphSAGE / GCN)',
            ha='center', va='center', fontsize=16, fontweight='bold', color='#2C3E50')
    ax.text(8, 5.6, 'Cada nodo agrega informacion de sus vecinos y actualiza su representacion',
            ha='center', va='center', fontsize=11, color='#7F8C8D')

    plt.tight_layout()
    plt.show()

draw_gnn_diagram()

## 15. Visualización 5 — Gráfico de barras comparativo

Comparación directa de Accuracy y F1-score entre:
- Baseline (Logistic Regression sobre TF-IDF)
- GraphSAGE (message passing sobre grafo k-NN)
- GCN (convolución espectral sobre grafo k-NN)

Para ambos casos de estudio.

In [ ]:
print('\n>>> VISUALIZACION 6: GRAFICO DE BARRAS COMPARATIVO <<<\n')

c1_acc_sage = all_results['C1_GraphSAGE']['best_acc']
c1_acc_gcn  = all_results['C1_GCN']['best_acc']
c2_acc_sage = all_results['C2_GraphSAGE']['best_acc']
c2_acc_gcn  = all_results['C2_GCN']['best_acc']

_, _, c1_f1_sage, _ = precision_recall_fscore_support(
    all_results['C1_GraphSAGE']['y_true'], all_results['C1_GraphSAGE']['y_pred'],
    average='macro')
_, _, c1_f1_gcn, _ = precision_recall_fscore_support(
    all_results['C1_GCN']['y_true'], all_results['C1_GCN']['y_pred'],
    average='macro')
_, _, c2_f1_sage, _ = precision_recall_fscore_support(
    all_results['C2_GraphSAGE']['y_true'], all_results['C2_GraphSAGE']['y_pred'],
    average='macro')
_, _, c2_f1_gcn, _ = precision_recall_fscore_support(
    all_results['C2_GCN']['y_true'], all_results['C2_GCN']['y_pred'],
    average='macro')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (case_name, baseline, sage_acc, gcn_acc, sage_f1, gcn_f1) in enumerate([
    ('Caso 1: Disaster Tweets', baseline_c1['acc'], c1_acc_sage, c1_acc_gcn,
     c1_f1_sage, c1_f1_gcn),
    ('Caso 2: AG News', baseline_c2['acc'], c2_acc_sage, c2_acc_gcn,
     c2_f1_sage, c2_f1_gcn),
]):
    ax = axes[ax_idx]
    x = np.arange(3)
    width = 0.35

    acc_values = [baseline, sage_acc, gcn_acc]

    bars_acc = ax.bar(x - width/2, acc_values, width, label='Accuracy',
                      color=['#95A5A6', '#2C3E50', '#3498DB'], edgecolor='black')

    # Need baseline F1 - compute it
    _, _, bl_f1, _ = precision_recall_fscore_support(
        baseline_c1['y_true'] if ax_idx == 0 else baseline_c2['y_true'],
        baseline_c1['y_pred'] if ax_idx == 0 else baseline_c2['y_pred'],
        average='macro')

    f1_values = [bl_f1, sage_f1, gcn_f1]
    bars_f1 = ax.bar(x + width/2, f1_values, width, label='F1-score (macro)',
                     color=['#BDC3C7', '#7F8C8D', '#85C1E9'], edgecolor='black')

    ax.set_xticks(x)
    ax.set_xticklabels(['Baseline\n(Log. Reg.)', 'GraphSAGE', 'GCN'], fontsize=10)
    ax.set_ylabel('Puntuacion', fontsize=11)
    ax.set_title(case_name, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars_acc, acc_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar, val in zip(bars_f1, f1_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Comparativa: Baseline vs GraphSAGE vs GCN',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 16. Dashboard interactivo — Demo de predicción en vivo

**Interfaz de usuario final del software SentinelGrafo.**

Permite:
1. Seleccionar el caso de estudio (Disaster Tweets o AG News)
2. Ingresar un texto libre en inglés
3. Predecir la clase con el modelo GraphSAGE entrenado
4. Visualizar el nivel de confianza de la predicción
5. Ver los 3 vecinos más similares del dataset de entrenamiento

**Cómo funciona:** El nuevo texto se limpia, se vectoriza con TF-IDF,
se conecta al grafo existente mediante k-NN, y se ejecuta el forward pass
de la GNN para obtener la predicción del nuevo nodo.


In [ ]:
# Limpiar la salida vieja del panel cuando se vuelve a ejecutar esta celda.
clear_output(wait=True)

# ===================================================================
# FUNCIONES DE AUXILIO E INFERENCIA
# ===================================================================
def predict_new_text(raw_text, model, data, vectorizer, tfidf_matrix, labels,
                     texts_original, k=5):
    '''Predice la clase de un nuevo texto usando la GNN entrenada.'''
    clean = clean_text(raw_text)
    if len(clean) < 3:
        return {
            'ok': False,
            'message': 'El texto es demasiado corto. Ingresa al menos 3 caracteres.'
        }

    new_vec = vectorizer.transform([clean]).toarray()
    sims = cosine_similarity(new_vec, tfidf_matrix)[0]
    neighbor_idx = np.argsort(sims)[-k:][::-1]

    new_x = torch.tensor(new_vec, dtype=torch.float)
    num_hops = get_message_passing_hops(model)
    local_x, local_edge_index, local_target_idx, local_context = build_local_prediction_subgraph(
        data, new_x, neighbor_idx, num_hops, labels
    )

    model.eval()
    with torch.no_grad():
        out = model(local_x.to(DEVICE), local_edge_index.to(DEVICE))
        probs = F.softmax(out[local_target_idx], dim=0).cpu().numpy()
        pred_class = int(probs.argmax())
        confidence = float(probs.max())

    neighbors_info = []
    for idx in neighbor_idx:
        similarity = float(sims[idx])
        label_str = str(labels[idx])
        snippet = texts_original[idx][:100] + '...' if len(texts_original[idx]) > 100 else texts_original[idx]
        neighbors_info.append({
            'dataset_idx': int(idx),
            'similarity': similarity,
            'label': label_str,
            'snippet': snippet
        })

    inference_meta = {
        'num_hops': num_hops,
        'subgraph_nodes': int(local_context['num_nodes']),
        'anchor_neighbors': int(len(neighbor_idx))
    }
    return {
        'ok': True,
        'pred_class': pred_class,
        'confidence': confidence,
        'probabilities': probs,
        'neighbors_info': neighbors_info,
        'inference_meta': inference_meta,
        'local_context': local_context
    }


def infer_case_data(case_key):
    if case_key == 'c1':
        return (all_results['C1_GraphSAGE']['model'], data_c1, vec_c1, tfidf_c1,
                np.array(labels_c1), df_c1['text'].tolist())
    return (all_results['C2_GraphSAGE']['model'], data_c2, vec_c2, tfidf_c2,
            np.array(labels_c2), df_c2['content'].tolist())


def get_case_visual_config(case_key):
    if case_key == 'c1':
        class_names = {0: 'NO DESASTRE', 1: 'DESASTRE REAL'}
        button_styles = {0: 'info', 1: 'danger'}
        border_colors = {0: '#3498DB', 1: '#E74C3C'}
    else:
        class_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
        button_styles = {0: 'info', 1: 'danger', 2: 'success', 3: 'warning'}
        border_colors = {0: '#3498DB', 1: '#E74C3C', 2: '#2ECC71', 3: '#F39C12'}
    return class_names, button_styles, border_colors


def get_message_passing_hops(model):
    hops = sum(1 for module in model.modules() if isinstance(module, (SAGEConv, GCNConv)))
    return max(1, hops)


def build_local_prediction_subgraph(data, new_x, neighbor_idx, num_hops, labels):
    neighbor_idx = np.array(neighbor_idx, dtype=int)
    anchor_set = set(neighbor_idx.tolist())
    new_node_idx = int(data.num_nodes)

    bridge_src = [new_node_idx] * len(neighbor_idx) + neighbor_idx.tolist()
    bridge_dst = neighbor_idx.tolist() + [new_node_idx] * len(neighbor_idx)
    bridge_edges = torch.tensor([bridge_src, bridge_dst], dtype=torch.long)
    augmented_edge_index = torch.cat([data.edge_index.cpu(), bridge_edges], dim=1)

    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        new_node_idx,
        num_hops,
        augmented_edge_index,
        relabel_nodes=True
    )

    base_x = data.x.cpu()
    local_rows = []
    local_labels = []
    anchor_local_indices = []
    original_indices = subset.tolist()

    for local_idx, node_idx in enumerate(original_indices):
        if node_idx == new_node_idx:
            local_rows.append(new_x[0])
            local_labels.append(-1)
        else:
            local_rows.append(base_x[node_idx])
            local_labels.append(int(labels[node_idx]))
            if node_idx in anchor_set:
                anchor_local_indices.append(local_idx)

    local_x = torch.stack(local_rows, dim=0)
    local_context = {
        'num_nodes': int(len(original_indices)),
        'original_indices': original_indices,
        'local_labels': local_labels,
        'anchor_local_indices': anchor_local_indices,
        'target_local_idx': int(mapping.item()),
        'local_edge_index': sub_edge_index.cpu()
    }
    return local_x, sub_edge_index, int(mapping.item()), local_context


def build_probability_widget(class_names, probs, border_colors, pred_class):
    rows = []
    for cls_idx in class_names.keys():
        label = widgets.Label(value=class_names[cls_idx], layout=widgets.Layout(width='140px'))
        bar = widgets.FloatProgress(
            value=float(probs[cls_idx]),
            min=0.0,
            max=1.0,
            layout=widgets.Layout(width='380px')
        )
        bar.style = {'bar_color': border_colors[cls_idx]}
        value_label = widgets.Label(
            value=f'{float(probs[cls_idx]):.1%}',
            layout=widgets.Layout(width='70px')
        )
        row = widgets.HBox([label, bar, value_label])
        if cls_idx == pred_class:
            row.layout = widgets.Layout(border='2px solid #1F2937', padding='4px', margin='2px 0')
        else:
            row.layout = widgets.Layout(border='1px solid #E5E7EB', padding='4px', margin='2px 0')
        rows.append(row)

    return widgets.VBox(
        rows,
        layout=widgets.Layout(border='1px solid #D9E2EC', padding='8px', width='100%')
    )


def build_prediction_graph_figure(case_key, prediction_result):
    _, _, border_colors = get_case_visual_config(case_key)
    pred_class = prediction_result['pred_class']
    local_context = prediction_result['local_context']
    edge_index = local_context['local_edge_index'].numpy()
    target_idx = local_context['target_local_idx']
    anchor_local = set(local_context['anchor_local_indices'])
    local_labels = local_context['local_labels']

    G = nx.Graph()
    G.add_nodes_from(range(local_context['num_nodes']))
    G.add_edges_from(zip(edge_index[0], edge_index[1]))

    fig, ax = plt.subplots(figsize=(7.8, 5.8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#F8FBFF')

    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No se pudo construir el subgrafo local de la prediccion.',
                ha='center', va='center', fontsize=11, color='#374151')
        ax.axis('off')
        return fig

    pos = nx.spring_layout(
        G,
        seed=42,
        k=1.4 / np.sqrt(max(G.number_of_nodes(), 1)),
        iterations=80
    )
    tx, ty = pos[target_idx]
    pos = {node: (coords[0] - tx, coords[1] - ty) for node, coords in pos.items()}

    highlighted_edges = []
    normal_edges = []
    for u, v in G.edges():
        if target_idx in (u, v):
            highlighted_edges.append((u, v))
        else:
            normal_edges.append((u, v))

    if normal_edges:
        nx.draw_networkx_edges(
            G, pos, edgelist=normal_edges, ax=ax,
            edge_color='#B9C3CF', alpha=0.35, width=1.0
        )
    if highlighted_edges:
        nx.draw_networkx_edges(
            G, pos, edgelist=highlighted_edges, ax=ax,
            edge_color=border_colors[pred_class], alpha=0.75, width=2.6
        )

    for cls_idx, color in border_colors.items():
        nodes = [i for i, lbl in enumerate(local_labels) if lbl == cls_idx and i != target_idx]
        if not nodes:
            continue
        halo_sizes = [320 if i in anchor_local else 170 for i in nodes]
        core_sizes = [250 if i in anchor_local else 125 for i in nodes]
        line_widths = [2.0 if i in anchor_local else 0.9 for i in nodes]
        edge_colors = ['#1F2937' if i in anchor_local else 'white' for i in nodes]
        nx.draw_networkx_nodes(
            G, pos, nodelist=nodes, ax=ax,
            node_color=color,
            node_size=halo_sizes,
            alpha=0.08,
            linewidths=0
        )
        nx.draw_networkx_nodes(
            G, pos, nodelist=nodes, ax=ax,
            node_color=color,
            node_size=core_sizes,
            alpha=0.92,
            edgecolors=edge_colors,
            linewidths=line_widths
        )

    nx.draw_networkx_nodes(
        G, pos, nodelist=[target_idx], ax=ax,
        node_color=border_colors[pred_class],
        node_size=620,
        node_shape='*',
        edgecolors='#111827',
        linewidths=1.9
    )

    ax.text(
        pos[target_idx][0],
        pos[target_idx][1] + 0.10,
        'Texto nuevo',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
        color='#111827'
    )

    for local_idx in list(anchor_local)[:3]:
        ax.text(
            pos[local_idx][0],
            pos[local_idx][1] - 0.08,
            'vecino ancla',
            ha='center',
            va='top',
            fontsize=8.5,
            color='#374151'
        )

    ax.set_title('Subgrafo local usado en la prediccion', fontsize=13, pad=12)
    ax.text(
        0.02, 0.02,
        'Estrella = texto nuevo | Nodos grandes = vecinos ancla',
        transform=ax.transAxes,
        fontsize=9,
        color='#4B5563',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#D1D5DB', alpha=0.9)
    )
    ax.axis('off')
    plt.tight_layout()
    return fig

def render_prediction_report(case_key, prediction_result):
    class_names, button_styles, border_colors = get_case_visual_config(case_key)
    pred_class = prediction_result['pred_class']
    confidence = prediction_result['confidence']
    inference_meta = prediction_result['inference_meta']
    neighbors_info = prediction_result['neighbors_info']

    explanation_text = (
        f'Se vectorizo el texto, se conecto con {inference_meta["anchor_neighbors"]} vecinos mas similares, '
        f'se extrajo un subgrafo local de {inference_meta["num_hops"]} saltos '
        f'con {inference_meta["subgraph_nodes"]} nodos y luego GraphSAGE realizo message passing para clasificarlo.'
    )

    main_color = border_colors[pred_class]
    result_card = widgets.VBox([
        widgets.Label(value='Resultado'),
        widgets.Button(
            description=class_names[pred_class],
            disabled=True,
            button_style=button_styles.get(pred_class, ''),
            layout=widgets.Layout(width='260px', height='38px')
        ),
        widgets.Label(value=f'Confianza: {confidence:.1%}'),
        widgets.Label(
            value=(
                f"Inferencia GNN local: {inference_meta['subgraph_nodes']} nodos | "
                f"{inference_meta['num_hops']} saltos | "
                f"{inference_meta['anchor_neighbors']} vecinos ancla"
            )
        )
    ], layout=widgets.Layout(
        border=f'2px solid {main_color}',
        padding='14px',
        margin='0 0 10px 0',
        width='100%'
    ))

    explanation_box = widgets.Box(
        [widgets.HTML(value=f'<div style="white-space: normal; line-height: 1.5;">{explanation_text}</div>')],
        layout=widgets.Layout(
            border='1px solid #D9E2EC',
            padding='12px',
            margin='0 0 10px 0',
            width='100%'
        )
    )

    probability_widget = build_probability_widget(
        class_names,
        prediction_result['probabilities'],
        border_colors,
        pred_class
    )

    sections = [
        result_card,
        explanation_box,
        widgets.Label(value='Probabilidades por clase'),
        probability_widget
    ]

    if isinstance(neighbors_info, list) and neighbors_info:
        sections.append(widgets.Label(value='Vecinos mas similares en el grafo'))
        for i, neighbor in enumerate(neighbors_info, start=1):
            raw_label = neighbor['label']
            neighbor_label = int(raw_label) if str(raw_label).isdigit() else raw_label
            if isinstance(neighbor_label, int):
                neighbor_label_text = class_names.get(neighbor_label, str(neighbor_label))
                neighbor_color = border_colors.get(neighbor_label, '#999999')
            else:
                neighbor_label_text = str(neighbor_label)
                neighbor_color = '#999999'

            snippet_box = widgets.Textarea(
                value=neighbor['snippet'],
                disabled=True,
                layout=widgets.Layout(width='100%', height='72px')
            )
            neighbor_card = widgets.VBox([
                widgets.Label(value=f'Vecino #{i} | similitud: {neighbor["similarity"]:.3f}'),
                widgets.Label(value=f'Etiqueta: {neighbor_label_text}'),
                snippet_box
            ], layout=widgets.Layout(
                border=f'2px solid {neighbor_color}',
                padding='10px',
                margin='4px 0',
                width='100%'
            ))
            sections.append(neighbor_card)

    set_prediction_widgets(sections)
    return build_prediction_graph_figure(case_key, prediction_result)


# Cerrar el panel anterior si la celda se ejecuta varias veces.
for widget_name in ['panel_title', 'tabs']:
    old_widget = globals().get(widget_name)
    if hasattr(old_widget, 'close'):
        old_widget.close()

panel_state = {
    'predict_busy': False,
    'graph_busy': False,
    'graph_cache': {}
}


# ===================================================================
# TAB 1: PREDICCION EN VIVO
# ===================================================================
case_selector = widgets.Dropdown(
    options=[
        ('Caso 1: Disaster Tweets', 'c1'),
        ('Caso 2: AG News', 'c2'),
    ],
    value='c1',
    description='Caso:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

text_input = widgets.Textarea(
    placeholder='Escribe un tweet o noticia en ingles para clasificar...',
    description='Texto:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%', height='100px')
)

predict_btn = widgets.Button(
    description='PREDECIR',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='150px', height='40px')
)

output_area = widgets.VBox(
    [],
    layout=widgets.Layout(width='92%', min_height='70px', margin='8px 0 0 0')
)
prediction_graph_output = widgets.Output(
    layout=widgets.Layout(width='92%', border='1px solid #D9E2EC', padding='6px', margin='10px 0 0 0')
)


def set_prediction_widgets(children):
    output_area.children = tuple(children)


def set_prediction_message(message, border_color='#D9D9D9'):
    message_box = widgets.Box(
        [widgets.Label(value=message)],
        layout=widgets.Layout(
            border=f'2px solid {border_color}',
            padding='12px',
            margin='0 0 8px 0',
            width='100%'
        )
    )
    set_prediction_widgets([message_box])
    with prediction_graph_output:
        clear_output(wait=True)


def on_predict_clicked(b):
    if panel_state['predict_busy']:
        return

    panel_state['predict_busy'] = True
    predict_btn.disabled = True

    try:
        set_prediction_widgets([])
        with prediction_graph_output:
            clear_output(wait=True)
        raw_text = text_input.value.strip()
        if not raw_text:
            set_prediction_message('Por favor ingresa un texto para clasificar.', border_color='#E74C3C')
            return

        model, data, vec, tfidf, labels, texts_orig = infer_case_data(case_selector.value)
        prediction_result = predict_new_text(
            raw_text, model, data, vec, tfidf, labels, texts_orig
        )

        if not prediction_result['ok']:
            set_prediction_message(prediction_result['message'], border_color='#E74C3C')
            return

        graph_fig = render_prediction_report(case_selector.value, prediction_result)
        with prediction_graph_output:
            clear_output(wait=True)
            print('Vecindario local utilizado por la GNN')
            display(graph_fig)
            plt.close(graph_fig)
    except Exception as exc:
        set_prediction_message(f'Error durante la prediccion: {exc}', border_color='#E74C3C')
    finally:
        predict_btn.disabled = False
        panel_state['predict_busy'] = False


predict_btn.on_click(on_predict_clicked, remove=True)
predict_btn.on_click(on_predict_clicked)

set_prediction_message('Escribe un texto y pulsa PREDECIR para ver el resultado.', border_color='#D9D9D9')

pred_tab_content = widgets.VBox([
    case_selector,
    text_input,
    predict_btn,
    output_area,
    prediction_graph_output
], layout=widgets.Layout(padding='10px'))


# ===================================================================
# TAB 2: EXPLORADOR DE GRAFOS (MATPLOTLIB)
# ===================================================================
graph_case_select = widgets.Dropdown(
    options=[
        ('Caso 1: Disaster Tweets', 'c1'),
        ('Caso 2: AG News (4 categorias)', 'c2'),
    ],
    value='c1',
    description='Caso:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

graph_view_mode_select = widgets.Dropdown(
    options=[
        ('Vista por clases', 'classes'),
        ('Vista por comunidades', 'communities'),
    ],
    value='classes',
    description='Colorear por:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

graph_highlight_class_select = widgets.Dropdown(
    options=[('Todas las clases', 'all')],
    value='all',
    description='Destacar clase:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

graph_nodes_slider = widgets.IntSlider(
    value=800, min=100, max=2000, step=100,
    description='Nodos:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px')
)

graph_k_slider = widgets.IntSlider(
    value=5, min=3, max=15, step=1,
    description='k-vecinos:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

graph_depth_slider = widgets.IntSlider(
    value=2, min=0, max=5, step=1,
    description='Capas BFS visibles:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px')
)

node_size_slider = widgets.IntSlider(
    value=8, min=3, max=25, step=1,
    description='Tamano nodo:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

edge_opacity_slider = widgets.FloatSlider(
    value=0.15, min=0.0, max=0.5, step=0.05,
    description='Opacidad aristas:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

graph_regen_btn = widgets.Button(
    description='Visualizar Grafo',
    button_style='info',
    icon='eye',
    layout=widgets.Layout(width='180px', height='36px')
)

graph_info_output = widgets.Output(layout=widgets.Layout(border='1px solid #D9D9D9', padding='8px'))
graph_output = widgets.Output()


def update_graph_info(message_lines):
    with graph_info_output:
        clear_output(wait=True)
        for line in message_lines:
            print(line)


def update_highlight_options(*_):
    case_key = graph_case_select.value
    class_names, _, _ = get_case_visual_config(case_key)
    options = [('Todas las clases', 'all')]
    options.extend((label, cls_idx) for cls_idx, label in class_names.items())
    current = graph_highlight_class_select.value
    graph_highlight_class_select.options = options
    valid_values = {value for _, value in options}
    graph_highlight_class_select.value = current if current in valid_values else 'all'


update_highlight_options()
graph_case_select.observe(update_highlight_options, names='value')


def get_graph_source(case_key, k_neighbors):
    cache_key = (case_key, k_neighbors)
    if cache_key in panel_state['graph_cache']:
        return panel_state['graph_cache'][cache_key]

    if case_key == 'c1':
        data_src = data_c1
        labels_src = labels_c1
        colors = ['#3498DB', '#E74C3C']
        class_labels = ['NO DESASTRE', 'DESASTRE REAL']
        title_base = 'Caso 1: Disaster Tweets'
        if k_neighbors != 5:
            data_src, _, _ = build_graph(texts_c1, labels_c1, max_features=3000, k=k_neighbors)
    else:
        data_src = data_c2
        labels_src = labels_c2
        colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']
        class_labels = ['World', 'Sports', 'Business', 'Sci/Tech']
        title_base = 'Caso 2: AG News - 4 categorias'
        if k_neighbors != 5:
            data_src, _, _ = build_graph(texts_c2, labels_c2, max_features=5000, k=k_neighbors)

    payload = (data_src, np.array(labels_src), colors, class_labels, title_base)
    panel_state['graph_cache'][cache_key] = payload
    return payload


def build_connected_subgraph(data_src, labels_src, n_nodes):
    from collections import defaultdict, deque

    n = min(n_nodes, data_src.num_nodes)
    edge_index = data_src.edge_index.cpu().numpy()
    adj = defaultdict(set)
    for u, v in zip(edge_index[0], edge_index[1]):
        if u == v:
            continue
        adj[int(u)].add(int(v))
        adj[int(v)].add(int(u))

    if not adj:
        idx_sample = list(range(n))
        root_global = 0
    else:
        root_global = max(adj.keys(), key=lambda node: len(adj[node]))
        selected_nodes = set()
        queue = deque([root_global])

        while queue and len(selected_nodes) < n:
            curr = queue.popleft()
            if curr in selected_nodes:
                continue
            selected_nodes.add(curr)
            for neighbor in sorted(adj[curr]):
                if neighbor not in selected_nodes:
                    queue.append(neighbor)

            if not queue and len(selected_nodes) < n:
                remaining = [node for node in range(data_src.num_nodes) if node not in selected_nodes]
                if remaining:
                    next_seed = max(remaining, key=lambda node: len(adj[node]))
                    queue.append(next_seed)

        idx_sample = sorted(selected_nodes)

    node_to_idx = {old: new for new, old in enumerate(idx_sample)}
    node_set = set(idx_sample)
    sub_src = []
    sub_dst = []

    for old_u in idx_sample:
        for old_v in adj.get(old_u, []):
            if old_v in node_set and old_u < old_v:
                sub_src.append(node_to_idx[old_u])
                sub_dst.append(node_to_idx[old_v])

    sub_edge_index = torch.tensor([sub_src + sub_dst, sub_dst + sub_src], dtype=torch.long)
    labels_sample = labels_src[idx_sample]
    sub_data = Data(
        x=data_src.x[idx_sample],
        edge_index=sub_edge_index,
        y=torch.tensor(labels_sample, dtype=torch.long)
    )
    root_local = node_to_idx.get(root_global, 0)
    return sub_data, idx_sample, labels_sample, root_local


def compute_community_map(graph):
    import matplotlib.colors as mcolors

    if graph.number_of_nodes() == 0:
        return {}, {}, 0
    if graph.number_of_edges() == 0 or graph.number_of_nodes() < 4:
        return {node: 0 for node in graph.nodes()}, {0: '#6B7280'}, 1

    communities = list(nx.community.greedy_modularity_communities(graph))
    palette = plt.colormaps['tab20'].resampled(max(len(communities), 1))
    color_map = {}
    node_to_community = {}
    for idx, community in enumerate(communities):
        color_map[idx] = mcolors.to_hex(palette(idx))
        for node in community:
            node_to_community[node] = idx
    return node_to_community, color_map, len(communities)


def build_graph_legend(ax, view_mode, class_labels, class_colors, community_colors, community_count):
    from matplotlib.lines import Line2D

    handles = [
        Line2D([0], [0], marker='*', color='w', markerfacecolor='#111827',
               markeredgecolor='#111827', markersize=12, label='Nodo semilla BFS'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
               markeredgecolor='#111827', markersize=9, label='Frontera BFS visible'),
    ]

    if view_mode == 'classes':
        for idx, label in enumerate(class_labels):
            handles.append(
                Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=class_colors[idx], markeredgecolor='white',
                       markersize=9, label=label)
            )
    else:
        visible_communities = sorted(community_colors.keys())[:6]
        for community_idx in visible_communities:
            handles.append(
                Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=community_colors[community_idx], markeredgecolor='white',
                       markersize=9, label=f'Comunidad {community_idx + 1}')
            )
        if community_count > 6:
            handles.append(mpatches.Patch(color='white', label=f'... y {community_count - 6} mas'))

    ax.legend(
        handles=handles,
        loc='upper right',
        frameon=True,
        facecolor='white',
        edgecolor='#D1D5DB',
        fontsize=9
    )


def plot_graph_overview(case_key, n_nodes, k_neighbors, node_size, edge_alpha,
                        view_mode='classes', highlighted_class='all', bfs_visible_layers=2):
    data_src, labels_src, colors, class_labels, title_base = get_graph_source(case_key, k_neighbors)
    sub_data, idx_sample, labels_sample, root_local = build_connected_subgraph(data_src, labels_src, n_nodes)

    G = to_networkx(sub_data, to_undirected=True)
    total_nodes = G.number_of_nodes()
    total_edges = G.number_of_edges()
    if total_nodes == 0:
        print('No hay nodos para visualizar.')
        return

    pos = nx.spring_layout(
        G,
        seed=42,
        k=1.15 / np.sqrt(max(total_nodes, 1)),
        iterations=60
    )

    bfs_levels = dict(nx.single_source_shortest_path_length(G, root_local))
    max_level_found = max(bfs_levels.values()) if bfs_levels else 0
    depth_limit = min(int(bfs_visible_layers), int(max_level_found))
    visible_nodes = [node for node, level in bfs_levels.items() if level <= depth_limit]
    if not visible_nodes:
        visible_nodes = [root_local]
        depth_limit = 0

    visible_G = G.subgraph(visible_nodes).copy()
    visible_edges = visible_G.number_of_edges()
    node_to_community, community_colors, community_count = compute_community_map(G)
    visible_frontier = {node for node in visible_nodes if bfs_levels.get(node, 0) == depth_limit and node != root_local}

    fig, ax = plt.subplots(figsize=(11.8, 8.8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#F7FBFF')

    edge_alpha_base = max(0.05, float(edge_alpha))
    nx.draw_networkx_edges(
        visible_G, pos, ax=ax,
        edge_color='#B7C1CC',
        alpha=edge_alpha_base,
        width=0.7
    )

    for node in visible_nodes:
        level = bfs_levels.get(node, depth_limit)
        label_idx = int(labels_sample[node])
        is_seed = node == root_local
        is_frontier = node in visible_frontier
        is_highlighted = highlighted_class == 'all' or label_idx == highlighted_class

        if view_mode == 'communities':
            color = community_colors.get(node_to_community.get(node, 0), '#6B7280')
        else:
            color = colors[label_idx]

        alpha = 0.95 if is_highlighted or is_seed else 0.14
        base_size = max(70, node_size * 12)
        size = base_size + max(0, 4 - level) * 18
        if is_seed:
            size = base_size * 3.0
        elif is_frontier:
            size = base_size * 1.55
        elif label_idx == highlighted_class:
            size = base_size * 1.18

        ax.scatter(
            pos[node][0], pos[node][1],
            s=size * 1.55,
            color=color,
            alpha=0.09 if is_highlighted or is_seed else 0.03,
            linewidths=0,
            zorder=2
        )
        ax.scatter(
            pos[node][0], pos[node][1],
            s=size,
            color=color,
            alpha=alpha,
            edgecolors='#111827' if is_seed or is_frontier else 'white',
            linewidths=1.8 if is_seed else (1.2 if is_frontier else 0.7),
            marker='*' if is_seed else 'o',
            zorder=3
        )

    title_suffix = 'Vista por clases' if view_mode == 'classes' else 'Vista por comunidades'
    ax.set_title(f'{title_base} ({total_nodes} nodos, k={k_neighbors}) - {title_suffix}', fontsize=15, pad=12, color='#1F2937')
    ax.text(
        0.02, 0.02,
        f'Capas BFS visibles: 0..{depth_limit} | Clase destacada: {"Todas" if highlighted_class == "all" else class_labels[highlighted_class]}',
        transform=ax.transAxes,
        fontsize=9.2,
        color='#4B5563',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#D1D5DB', alpha=0.92)
    )
    ax.axis('off')
    build_graph_legend(ax, view_mode, class_labels, colors, community_colors, community_count)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

    avg_degree = (2 * total_edges / total_nodes) if total_nodes else 0.0
    highlight_label = 'Todas las clases' if highlighted_class == 'all' else class_labels[highlighted_class]
    view_label = 'coloreado por clases' if view_mode == 'classes' else 'coloreado por comunidades'
    update_graph_info([
        'Resumen visual: estas viendo un subgrafo conectado muestreado desde un nodo semilla BFS.',
        f'Vista actual: {view_label} | Clase destacada: {highlight_label} | Comunidades detectadas: {community_count}.',
        f'Nodos visibles: {len(visible_nodes)}/{total_nodes} | Aristas visibles: {visible_edges}/{total_edges} | Grado promedio global: {avg_degree:.1f}.',
        f'Lectura sugerida: el nodo semilla marca el origen del crecimiento y la frontera muestra la ultima capa BFS visible.'
    ])


def on_graph_regen(b):
    if panel_state['graph_busy']:
        return

    panel_state['graph_busy'] = True
    graph_regen_btn.disabled = True

    with graph_output:
        clear_output(wait=True)
        try:
            case = graph_case_select.value
            n_nodes = graph_nodes_slider.value
            k_val = graph_k_slider.value
            n_size = node_size_slider.value
            e_alpha = edge_opacity_slider.value
            view_mode = graph_view_mode_select.value
            highlighted_class = graph_highlight_class_select.value
            depth_limit = graph_depth_slider.value

            if k_val != 5:
                print(f'Reconstruyendo grafo con k={k_val}... puede tardar unos segundos.')

            plot_graph_overview(
                case, n_nodes, k_val, n_size, e_alpha,
                view_mode=view_mode,
                highlighted_class=highlighted_class,
                bfs_visible_layers=depth_limit
            )
        except Exception as exc:
            print(f'No se pudo generar el grafo: {exc}')
            update_graph_info(['No hay estadisticas disponibles porque la visualizacion fallo.'])
        finally:
            graph_regen_btn.disabled = False
            panel_state['graph_busy'] = False


graph_regen_btn.on_click(on_graph_regen, remove=True)
graph_regen_btn.on_click(on_graph_regen)

update_graph_info([
    'Selecciona una vista, decide cuantas capas BFS quieres mostrar y luego pulsa Visualizar Grafo.',
    'La opcion "Destacar clase" atenúa el resto para que puedas leer mejor la estructura.'
])

graph_tab_content = widgets.VBox([
    widgets.HBox([graph_case_select, graph_view_mode_select, graph_highlight_class_select]),
    widgets.HBox([graph_nodes_slider, graph_k_slider]),
    widgets.HBox([graph_depth_slider, node_size_slider]),
    widgets.HBox([edge_opacity_slider, graph_regen_btn]),
    graph_info_output,
    graph_output
], layout=widgets.Layout(padding='10px'))

with graph_output:
    try:
        plot_graph_overview(
            'c1', 800, 5, 8, 0.15,
            view_mode='classes',
            highlighted_class='all',
            bfs_visible_layers=2
        )
    except Exception as exc:
        print(f'No se pudo cargar el grafo inicial: {exc}')
        update_graph_info(['No hay estadisticas disponibles porque la visualizacion inicial fallo.'])

tabs = widgets.Tab()
tabs.children = [pred_tab_content, graph_tab_content]
tabs.set_title(0, 'Prediccion en Vivo')
tabs.set_title(1, 'Explorador de Grafos')

panel_title = widgets.Label(value='SentinelGrafo - Panel Interactivo')
display(panel_title)
display(tabs)


## 17. Tabla resumen final de resultados

Se consolidan todos los resultados en un DataFrame para comparación directa.
Incluye Baseline, GraphSAGE y GCN para ambos casos de estudio.

In [ ]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    return acc, prec, rec, f1


rows = []

for case_name, baseline, sage_key, gcn_key in [
    ('Caso 1: Disaster Tweets', baseline_c1, 'C1_GraphSAGE', 'C1_GCN'),
    ('Caso 2: AG News', baseline_c2, 'C2_GraphSAGE', 'C2_GCN'),
]:
    rows.append({
        'Caso': case_name,
        'Modelo': 'Baseline (Log. Reg.)',
        'Accuracy': f"{baseline['acc']:.4f}",
        'Precision': f"{baseline['prec']:.4f}",
        'Recall': f"{baseline['rec']:.4f}",
        'F1-score': f"{baseline['f1']:.4f}",
    })

    for model_name, key in [('GraphSAGE', sage_key), ('GCN', gcn_key)]:
        r = all_results[key]
        acc, prec, rec, f1 = compute_metrics(r['y_true'], r['y_pred'])
        rows.append({
            'Caso': case_name,
            'Modelo': model_name,
            'Accuracy': f"{acc:.4f}",
            'Precision': f"{prec:.4f}",
            'Recall': f"{rec:.4f}",
            'F1-score': f"{f1:.4f}",
        })

df_results = pd.DataFrame(rows)

def highlight_best(val, row, col):
    # Simple styling
    return ''

styled = df_results.style.set_caption('RESULTADOS FINALES — SentinelGrafo').set_properties(**{
    'text-align': 'center'
}).set_table_styles([
    dict(selector='caption', props=[('font-size', '16px'), ('font-weight', 'bold'),
                                    ('text-align', 'center'), ('padding', '10px')]),
    dict(selector='th', props=[('background-color', '#2C3E50'), ('color', 'white'),
                                ('font-weight', 'bold'), ('text-align', 'center')]),
    dict(selector='td', props=[('text-align', 'center'), ('padding', '8px')]),
])

display(styled)

print('\n>>> NOTA: Todas las metricas estan calculadas sobre el conjunto de TEST. <<<')
print('El grafo se construyo con k=5 vecinos y TF-IDF de 3000 features.')
print('Modelos GNN entrenados con 2 capas, 64 dimensiones ocultas y early stopping.')


---

## Software Documentado — Fin del Anexo

Este documento constituye el software documentado completo de **SentinelGrafo**,
desarrollado como parte del Trabajo Computacional Nº 1 del curso
Inteligencia Artificial 2026-1, Sección 3.

**Grupo Segovia — Integrantes:**
- Segovia Valencia Jim Bryan
- Morales Usca Andres
- Chavez Cerna Joshua

**Tema:** Redes Neuronales Gráficas: fundamentos, aplicación y casos de estudio

---